# گام اول: بهبود کد کلاسی (ارتقا به شبکه ۲ بعدی با هزینه انرژی)
### آزمایشگاه هوش مصنوعی — پیاده‌سازی و مقایسه عامل‌های هوشمند
**پروژه:** مسیریابی هوشمند شهری (Urban Route Finding)
**اعضا:** ایلیا آزادپور (نماینده)، سپهر صابری (پیاده‌سازی)، قائدیان، سینا محمدپورپورانی

---
### 🌟 اهداف بهبود کد کلاسی:
در کلاس درس، عامل جاروبرقی روی یک خط تک‌بعدی (۴ اتاق خطی) پیاده شده بود. برای هم‌راستایی با پروژه **مسیریابی شهری**، تغییرات زیر اعمال شد:
1. **محیط دوبعدی ($N \times N$ Grid):** مدل‌سازی محیط به شکل بلوک‌های دوبعدی شهر (حرکت در ۴ جهت اصلی Up/Down/Left/Right).
2. **هزینه انرژی و سوخت (Energy / Fuel Cost):** هر حرکت معادل ۱ واحد مصرف انرژی و پاک‌سازی معادل ۲ واحد است؛ توقف (NoOp) انرژی مصرف نمی‌کند.
3. **عامل مدل‌محور با حافظه فضایی (Smart Spatial Memory):** عامل علاوه بر وضعیت کنونی، نقشه خانه‌های کاوش‌شده را نگهداری می‌کند و با تخمین فاصله منهتن، به سمت نزدیک‌ترین هدف حرکت می‌کند و با اتمام مأموریت متوقف می‌شود.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# تنظیمات محیط شبیه‌سازی ۲ بعدی
# ==========================================
GRID_SIZE = 3      # ابعاد محیط ۳ در ۳ (شبیه بلوک‌های شهری)
STEPS = 40         # تعداد گام‌های زمانی در هر آزمایش
RUNS = 100         # تعداد دفعات تکرار آزمایش
ACTIONS = ["Up", "Down", "Left", "Right", "Clean", "NoOp"]

class GridEnvironment:
    """محیط شبکه دوبعدی با کثیفی تصادفی و هزینه مصرف انرژی"""
    def __init__(self, size=GRID_SIZE, seed=None):
        self.size = size
        self.rng = np.random.default_rng(seed)
        # هر خانه با احتمال 40 درصد کثیف/دارای مانع است
        self.grid = self.rng.random((size, size)) < 0.4
        self.total_dirt = int(self.grid.sum())

    def is_dirty(self, r, c):
        return self.grid[r, c]

    def clean(self, r, c):
        if self.grid[r, c]:
            self.grid[r, c] = False
            return True
        return False

    def count_clean(self):
        return int((~self.grid).sum())

    def is_all_clean(self):
        return not self.grid.any()


def simulate_agent(agent_type, seed):
    """
    اجرای شبیه‌سازی برای یکی از سه عامل:
    1. random: انتخاب کاملاً تصادفی عمل‌ها
    2. reflex: اگر خانه کثیف بود تمیز کن، وگرنه حرکت تصادفی در جهت مجاز
    3. smart_model: عامل مبتنی بر مدل با حافظه فضایی دوبعدی؛ یافتن کوتاه‌ترین مسیر به خانه‌های کثیف/دیده نشده و توقف با NoOp
    """
    env = GridEnvironment(size=GRID_SIZE, seed=seed)
    rng = np.random.default_rng(seed)
    
    # موقعیت شروع در گوشه (0, 0)
    r, c = 0, 0
    energy = 100       # ذخیره انرژی اولیه (باک سوخت)
    score = 0          # امتیاز کسب‌شده
    
    # حافظه عامل مدل‌محور (ثبت وضعیت دیده‌شده هر خانه)
    # 0 = ناشناخته، 1 = تمیز، 2 = کثیف
    memory = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
    
    for step in range(STEPS):
        # 1. مشاهده وضعیت فعلی و به‌روزرسانی حافظه
        currently_dirty = env.is_dirty(r, c)
        memory[r, c] = 2 if currently_dirty else 1
        
        action = "NoOp"
        
        # 2. تصمیم‌گیری بر اساس نوع عامل
        if agent_type == "random":
            action = rng.choice(ACTIONS)
            
        elif agent_type == "reflex":
            if currently_dirty:
                action = "Clean"
            else:
                action = rng.choice(["Up", "Down", "Left", "Right"])
                
        elif agent_type == "smart_model":
            if currently_dirty:
                action = "Clean"
            elif (memory == 1).all() or env.is_all_clean():
                # وقتی همه خانه‌ها تمیز شناخته شدند، متوقف شو تا انرژی هدر نرود
                action = "NoOp"
            else:
                # جستجوی نزدیک‌ترین خانه کثیف یا ناشناخته (ایده اولیه مسیریابی)
                best_dist = 999
                target = None
                for tr in range(GRID_SIZE):
                    for tc in range(GRID_SIZE):
                        if memory[tr, tc] != 1:  # هنوز تمیز نشده
                            dist = abs(tr - r) + abs(tc - c)
                            if dist < best_dist:
                                best_dist = dist
                                target = (tr, tc)
                                
                if target:
                    tr, tc = target
                    if tr > r: action = "Down"
                    elif tr < r: action = "Up"
                    elif tc > c: action = "Right"
                    elif tc < c: action = "Left"
                else:
                    action = "NoOp"
                    
        # 3. اعمال عمل و محاسبه هزینه انرژی
        if action == "Clean":
            env.clean(r, c)
            memory[r, c] = 1
            energy -= 2   # هزینه انرژی پاک‌سازی
            score += 15   # پاداش تمیز کردن
        elif action == "Up":
            if r > 0: r -= 1
            energy -= 1
        elif action == "Down":
            if r < GRID_SIZE - 1: r += 1
            energy -= 1
        elif action == "Left":
            if c > 0: c -= 1
            energy -= 1
        elif action == "Right":
            if c < GRID_SIZE - 1: c += 1
            energy -= 1
        elif action == "NoOp":
            pass  # بدون مصرف انرژی اضافه
            
        # پاداش ماندگاری در محیط تمیز (+1 به ازای هر خانه تمیز در هر گام)
        score += env.count_clean()
        
    return {"score": score, "remaining_energy": max(0, energy)}


## ۲. اجرای آزمایش روی ۱۰۰ تکرار و مقایسه عملکرد
هر یک از سه عامل (Random، Reflex، و Smart Model-based) تحت ۱۰۰ سنجه تصادفی یکسان ارزیابی می‌شوند:

In [ ]:
records = []
agents = ["random", "reflex", "smart_model"]

for agent in agents:
    for seed in range(RUNS):
        res = simulate_agent(agent, seed)
        records.append({
            "Agent": agent,
            "Score": res["score"],
            "Remaining_Energy": res["remaining_energy"]
        })

df = pd.DataFrame(records)

# نمایش جدول آماری میانگین و انحراف معیار
summary = df.groupby("Agent").agg({
    "Score": ["mean", "std"],
    "Remaining_Energy": ["mean", "std"]
}).round(1)

print("=" * 65)
print("           جدول نتایج آماری عملکرد عامل‌ها (۱۰۰ بار اجرا)")
print("=" * 65)
display(summary)

## ۳. تحلیل نموداری نتایج
رسم مقایسه‌ای امتیاز نهایی و مصرف بهینه انرژی بین سه عامل:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# نمودار میانگین امتیاز
mean_scores = df.groupby("Agent")["Score"].mean()
std_scores = df.groupby("Agent")["Score"].std()
axes[0].bar(mean_scores.index, mean_scores.values, yerr=std_scores.values, capsize=6, color=["#94a3b8", "#38bdf8", "#4f46e5"])
axes[0].set_title("Average Performance Score (Higher is better)")
axes[0].set_ylabel("Score")
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

# نمودار ذخیره انرژی باقیمانده
mean_energy = df.groupby("Agent")["Remaining_Energy"].mean()
std_energy = df.groupby("Agent")["Remaining_Energy"].std()
axes[1].bar(mean_energy.index, mean_energy.values, yerr=std_energy.values, capsize=6, color=["#f87171", "#fbbf24", "#34d399"])
axes[1].set_title("Average Remaining Energy (Fuel Efficiency)")
axes[1].set_ylabel("Energy Units")
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle("Comparison of Vacuum/Urban Agents in 2D Grid Environment", fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

## ۴. نتیجه‌گیری و ارتباط با پروژه مسیریابی شهری
1. **برتری قاطع عامل مدل‌محور هوشمند:** عامل `smart_model` با داشتن نقشه شناختی از محیط دوبعدی، مستقیماً به سمت خانه‌های هدف حرکت کرده و پس از اتمام کار، با انتخاب `NoOp` مانع از هدررفت انرژی می‌شود.
2. **پیوند با گام‌های بعدی:** منطق محاسبه فاصله منهتن و انتخاب گره هدف، پایه‌ی اولیه الگوریتم‌های ناوبری نظیر $A^*$ در شبکه خیابان‌های شهری خواهد بود که در گام‌های ۴ و ۵ تکمیل می‌شود.